# Election Time Series — U.S. House Results

This notebook builds a time series of U.S. House election results for Travis County,
interpolated onto the 2026 congressional district boundaries using the population
weights calculated in notebook 03.

## Data sources
- 2016: VEST (tx_2016.zip)
- 2018: VEST (tx_2018.zip)  
- 2020: VEST (tx_2020.zip)
- 2022: TBD
- 2024: TBD

## Goal
Produce a single time series file: /data/processed/travis_house_time_series.csv
Columns: year, new_district_id, candidate, party, estimated_votes

In [1]:
import geopandas as gpd
import pandas as pd
import zipfile

# Load processed files from Phase 1
precincts = gpd.read_file('../data/processed/travis_precincts_2020.gpkg')
weights = pd.read_csv('../data/processed/travis_population_weights.csv')

print(f"Precincts: {len(precincts)}")
print(f"Weights table: {len(weights)} rows")
print(f"Weights columns: {list(weights.columns)}")

Precincts: 247
Weights table: 418 rows
Weights columns: ['old_precinct_id', 'new_district_id', 'fragment_population', 'weight']


In [2]:
# Load 2016 VEST shapefile
tx_2016 = gpd.read_file('zip://../data/raw/election_results/tx_2016.zip!tx_2016.shp')
print(f"Total statewide precincts: {len(tx_2016)}")
print(f"CRS: {tx_2016.crs}")

# Filter to Travis County
travis_2016 = tx_2016[tx_2016['CNTY'] == 453].copy()
print(f"\nTravis County precincts: {len(travis_2016)}")
print(f"\nColumns: {list(travis_2016.columns)}")
print(travis_2016.head(3))

Total statewide precincts: 8832
CRS: PROJCS["NAD_1983_Lambert_Conformal_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AUTHORITY["EPSG","4269"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",31.1666666666667],PARAMETER["central_meridian",-100],PARAMETER["standard_parallel_1",27.4166666666667],PARAMETER["standard_parallel_2",34.9166666666667],PARAMETER["false_easting",1000000],PARAMETER["false_northing",1000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]

Travis County precincts: 247

Columns: ['CNTY', 'COLOR', 'PREC', 'PCTKEY', 'CNTYKEY', 'G16VR', 'G16SSVR', 'G16PRERTRU', 'G16PREDCLI', 'G16PRELJOH', 'G16PREGSTE', 'G16PREOWRI', 'G16RRCRCHR', 'G16RRCDYAR', 'G16RRCLMIL', 'G16RRCGSAL', 'G16SSCRLEH', 'G16SSCDWES', 'G16SSCLGLA', 'G16SSCGMUN', 'G16SSCRGRE', 'G16SSCDCO

In [3]:
# Find U.S. House columns
house_cols = [c for c in travis_2016.columns if 'USH' in c or 'USS' in c or 'HOU' in c]
print("Possible House columns:", house_cols)

# Also print all column names to scan manually
for col in travis_2016.columns:
    print(col)

Possible House columns: []
CNTY
COLOR
PREC
PCTKEY
CNTYKEY
G16VR
G16SSVR
G16PRERTRU
G16PREDCLI
G16PRELJOH
G16PREGSTE
G16PREOWRI
G16RRCRCHR
G16RRCDYAR
G16RRCLMIL
G16RRCGSAL
G16SSCRLEH
G16SSCDWES
G16SSCLGLA
G16SSCGMUN
G16SSCRGRE
G16SSCDCON
G16SSCLOXF
G16SSCGWAT
G16SSCRGUZ
G16SSCDROB
G16SSCLFUL
G16SSCGCHI
G16SCCRKEE
G16SCCDMEY
G16SCCLASH
G16SCCGREP
G16SCCRWAL
G16SCCDJOH
G16SCCLSTR
G16SCCGSAN
G16SCCRKEA
G16SCCDBUR
G16SCCLBEN
geometry


In [4]:
# Check the documentation
with zipfile.ZipFile('../data/raw/election_results/tx_2016.zip', 'r') as z:
    print(z.namelist())

['tx_2016.shx', 'tx_2016.cpg', 'tx_2016.dbf', 'tx_2016.prj', 'tx_2016.shp']


In [5]:
# Check the 2020 file columns for comparison — we know this one works
tx_2020 = gpd.read_file('zip://../data/raw/election_results/tx_2020.zip!tx_2020.shp')
travis_2020 = tx_2020[tx_2020['CNTY'] == 453].copy()

# Find any House columns
house_cols_2020 = [c for c in travis_2020.columns if 'USH' in c or 'CON' in c]
print("2020 House columns:", house_cols_2020[:20])
print(f"\nTotal 2020 columns: {len(travis_2020.columns)}")

2020 House columns: []

Total 2020 columns: 38


# Testing again, troubleshooting

In [6]:
import geopandas as gpd
import pandas as pd

# Load the 2020 precinct boundary file
precincts = gpd.read_file('../data/processed/travis_precincts_2020.gpkg')

print(f"Total precincts: {len(precincts)}")
print(f"\nPCTKEY sample:")
print(sorted(precincts['PCTKEY'].values)[:20])
print(f"\nPCTKEY dtype: {precincts['PCTKEY'].dtype}")

Total precincts: 247

PCTKEY sample:
['4530101', '4530102', '4530103', '4530104', '4530105', '4530106', '4530107', '4530108', '4530109', '4530110', '4530111', '4530112', '4530113', '4530114', '4530115', '4530116', '4530117', '4530118', '4530119', '4530120']

PCTKEY dtype: object


In [7]:
# Check all PKTKEYs start with 453
all_start_453 = precincts['PCTKEY'].str.startswith('453').all()
print(f"All start with 453: {all_start_453}")

# Check length consistency
lengths = precincts['PCTKEY'].str.len().value_counts()
print(f"\nPCTKEY lengths:\n{lengths}")

# Any unexpected values?
unusual = precincts[~precincts['PCTKEY'].str.startswith('453')]
print(f"\nPrecincts not starting with 453: {len(unusual)}")

All start with 453: True

PCTKEY lengths:
7    247
Name: PCTKEY, dtype: int64

Precincts not starting with 453: 0


In [8]:
# Load VEST 2020
tx_2020 = gpd.read_file('zip://../data/raw/election_results/tx_2020.zip!tx_2020.shp')
travis_2020 = tx_2020[tx_2020['CNTY'] == 453].copy()

print(f"Travis precincts in VEST 2020: {len(travis_2020)}")
print(f"\nPCTKEY sample:")
print(sorted(travis_2020['PCTKEY'].values)[:20])

# Check match against boundary file
matches = set(travis_2020['PCTKEY']) & set(precincts['PCTKEY'])
only_vest = set(travis_2020['PCTKEY']) - set(precincts['PCTKEY'])
only_boundary = set(precincts['PCTKEY']) - set(travis_2020['PCTKEY'])

print(f"\nMatching: {len(matches)}")
print(f"Only in VEST 2020: {len(only_vest)}")
print(f"Only in boundary: {len(only_boundary)}")

Travis precincts in VEST 2020: 247

PCTKEY sample:
['4530101', '4530102', '4530103', '4530104', '4530105', '4530106', '4530107', '4530108', '4530109', '4530110', '4530111', '4530112', '4530113', '4530114', '4530115', '4530116', '4530117', '4530118', '4530119', '4530120']

Matching: 247
Only in VEST 2020: 0
Only in boundary: 0


In [9]:
# Load VEST 2016
tx_2016 = gpd.read_file('zip://../data/raw/election_results/tx_2016.zip!tx_2016.shp')
travis_2016 = tx_2016[tx_2016['CNTY'] == 453].copy()

print(f"Travis precincts in VEST 2016: {len(travis_2016)}")

matches_2016 = set(travis_2016['PCTKEY']) & set(precincts['PCTKEY'])
only_vest_2016 = set(travis_2016['PCTKEY']) - set(precincts['PCTKEY'])
only_boundary_2016 = set(precincts['PCTKEY']) - set(travis_2016['PCTKEY'])

print(f"Matching: {len(matches_2016)}")
print(f"Only in VEST 2016: {len(only_vest_2016)}")
print(f"Only in boundary: {len(only_boundary_2016)}")

# Load VEST 2018
tx_2018 = gpd.read_file('zip://../data/raw/election_results/tx_2018.zip!tx_2018.shp')
travis_2018 = tx_2018[tx_2018['CNTY'] == 453].copy()

print(f"\nTravis precincts in VEST 2018: {len(travis_2018)}")

matches_2018 = set(travis_2018['PCTKEY']) & set(precincts['PCTKEY'])
only_vest_2018 = set(travis_2018['PCTKEY']) - set(precincts['PCTKEY'])
only_boundary_2018 = set(precincts['PCTKEY']) - set(travis_2018['PCTKEY'])

print(f"Matching: {len(matches_2018)}")
print(f"Only in VEST 2018: {len(only_vest_2018)}")
print(f"Only in boundary: {len(only_boundary_2018)}")

Travis precincts in VEST 2016: 247
Matching: 247
Only in VEST 2016: 0
Only in boundary: 0

Travis precincts in VEST 2018: 247
Matching: 247
Only in VEST 2018: 0
Only in boundary: 0


In [10]:
# Check all three for House columns
for year, df in [('2016', travis_2016), ('2018', travis_2018), ('2020', travis_2020)]:
    house_cols = [c for c in df.columns if 'USH' in c or 'CON' in c or 'HOU' in c]
    all_cols = [c for c in df.columns if c not in ['CNTY', 'COLOR', 'PREC', 'PCTKEY', 'CNTYKEY', 'geometry']]
    print(f"\nVEST {year}:")
    print(f"  House columns found: {house_cols}")
    print(f"  All race columns: {all_cols}")


VEST 2016:
  House columns found: ['G16SSCDCON']
  All race columns: ['G16VR', 'G16SSVR', 'G16PRERTRU', 'G16PREDCLI', 'G16PRELJOH', 'G16PREGSTE', 'G16PREOWRI', 'G16RRCRCHR', 'G16RRCDYAR', 'G16RRCLMIL', 'G16RRCGSAL', 'G16SSCRLEH', 'G16SSCDWES', 'G16SSCLGLA', 'G16SSCGMUN', 'G16SSCRGRE', 'G16SSCDCON', 'G16SSCLOXF', 'G16SSCGWAT', 'G16SSCRGUZ', 'G16SSCDROB', 'G16SSCLFUL', 'G16SSCGCHI', 'G16SCCRKEE', 'G16SCCDMEY', 'G16SCCLASH', 'G16SCCGREP', 'G16SCCRWAL', 'G16SCCDJOH', 'G16SCCLSTR', 'G16SCCGSAN', 'G16SCCRKEA', 'G16SCCDBUR', 'G16SCCLBEN']

VEST 2018:
  House columns found: []
  All race columns: ['G18VR', 'G18SSVR', 'G18USSRCRU', 'G18USSDORO', 'G18USSLDIK', 'G18GOVRABB', 'G18GOVDVAL', 'G18GOVLTIP', 'G18LTGRPAT', 'G18LTGDCOL', 'G18LTGLMCK', 'G18ATGRPAX', 'G18ATGDNEL', 'G18ATGLHAR', 'G18COMRHEG', 'G18COMDCHE', 'G18COMLSAN', 'G18LNDRBUS', 'G18LNDDSUA', 'G18LNDLPIN', 'G18AGRRMIL', 'G18AGRDOLS', 'G18AGRLCAR', 'G18RRCRCRA', 'G18RRCDMCA', 'G18RRCLWRI', 'G18SSCRBLA', 'G18SSCDKIR', 'G18SSCRDEV', 'G18

In [11]:
# Load MEDSL 2020 House file
medsl = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general.csv',
                    dtype={'precinct': str, 'county_fips': str})

# Filter to Texas and Travis County
tx_house = medsl[medsl['state_po'] == 'TX'].copy()
travis_house = tx_house[tx_house['county_name'] == 'TRAVIS'].copy()

print(f"Texas rows: {len(tx_house)}")
print(f"Travis rows: {len(travis_house)}")
print(f"\nSample precinct values:")
print(travis_house['precinct'].head(10).tolist())

Texas rows: 29845
Travis rows: 834

Sample precinct values:
['4530221', '4530221', '4530231', '4530231', '4530231', '4530232', '4530232', '4530232', '4530233', '4530233']


In [12]:
# Check unique precinct matches
travis_unique = set(travis_house['precinct'].unique())
boundary_unique = set(precincts['PCTKEY'].values)

in_both = travis_unique & boundary_unique
only_medsl = travis_unique - boundary_unique
only_boundary = boundary_unique - travis_unique

print(f"Unique precincts in MEDSL Travis: {len(travis_unique)}")
print(f"Matching: {len(in_both)}")
print(f"Only in MEDSL: {len(only_medsl)}")
print(f"Only in boundary: {len(only_boundary)}")

# Check candidates and districts
print(f"\nCandidates:")
print(travis_house[['candidate', 'party_simplified', 'district']].drop_duplicates())

Unique precincts in MEDSL Travis: 247
Matching: 247
Only in MEDSL: 0
Only in boundary: 0

Candidates:
            candidate party_simplified  district
509352         KELSEY      LIBERTARIAN        25
509353       WILLIAMS       REPUBLICAN        25
509354         OLIVER         DEMOCRAT        25
525413         SIEGEL         DEMOCRAT        10
525414        ERIKSEN      LIBERTARIAN        10
525415         MCCAUL       REPUBLICAN        10
531412        KENNEDY         DEMOCRAT        17
531413          BROWN      LIBERTARIAN        17
531414       SESSIONS       REPUBLICAN        17
535400          DAVIS         DEMOCRAT        21
535401       DIBIANCA      LIBERTARIAN        21
535402         WAKELY            OTHER        21
535403            ROY       REPUBLICAN        21
546997        DOGGETT         DEMOCRAT        35
546998          LOEWE      LIBERTARIAN        35
546999           MATA            OTHER        35
547000  GARCIA SHARON       REPUBLICAN        35


In [15]:
df_2016 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2016.tab', 
                       sep='\t', dtype={'precinct': str})
print(df_2016.columns.tolist())
print(df_2016.head(2))

['year,stage,special,state,state_postal,state_fips,state_icpsr,county_name,county_fips,county_ansi,county_lat,county_long,jurisdiction,precinct,candidate,candidate_normalized,office,district,writein,party,mode,votes,candidate_opensecrets,candidate_wikidata,candidate_party,candidate_last,candidate_first,candidate_middle,candidate_full,candidate_suffix,candidate_nickname,candidate_fec,candidate_fec_name,candidate_google,candidate_govtrack,candidate_icpsr,candidate_maplight']
  year,stage,special,state,state_postal,state_fips,state_icpsr,county_name,county_fips,county_ansi,county_lat,county_long,jurisdiction,precinct,candidate,candidate_normalized,office,district,writein,party,mode,votes,candidate_opensecrets,candidate_wikidata,candidate_party,candidate_last,candidate_first,candidate_middle,candidate_full,candidate_suffix,candidate_nickname,candidate_fec,candidate_fec_name,candidate_google,candidate_govtrack,candidate_icpsr,candidate_maplight
0  2016,gen,False,Alabama,AL,1,41,Autauga Coun

In [16]:
df_2016 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2016.tab', 
                       sep=',', dtype={'precinct': str, 'county_fips': str})
print(df_2016.columns.tolist())

/Users/flyingcape/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3165: DtypeWarning: Columns (29,30) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


['year', 'stage', 'special', 'state', 'state_postal', 'state_fips', 'state_icpsr', 'county_name', 'county_fips', 'county_ansi', 'county_lat', 'county_long', 'jurisdiction', 'precinct', 'candidate', 'candidate_normalized', 'office', 'district', 'writein', 'party', 'mode', 'votes', 'candidate_opensecrets', 'candidate_wikidata', 'candidate_party', 'candidate_last', 'candidate_first', 'candidate_middle', 'candidate_full', 'candidate_suffix', 'candidate_nickname', 'candidate_fec', 'candidate_fec_name', 'candidate_google', 'candidate_govtrack', 'candidate_icpsr', 'candidate_maplight']


In [17]:
files = {
    '2016': ('../data/raw/election_results/HOUSE_precinct_general_2016.tab', ','),
    '2018': ('../data/raw/election_results/HOUSE_precinct_general_2018.csv', ','),
    '2022': ('../data/raw/election_results/HOUSE_precinct_general_2022.csv', ','),
    '2024': ('../data/raw/election_results/HOUSE_precinct_general_2024.csv', ','),
}

for year, (path, sep) in files.items():
    df = pd.read_csv(path, sep=sep, dtype={'precinct': str, 'county_fips': str})
    
    # Handle different column names across years
    county_col = 'county_name' if 'county_name' in df.columns else 'county_name'
    state_col = 'state_po' if 'state_po' in df.columns else 'state_postal'
    
    travis = df[df['county_name'] == 'TRAVIS'].copy()
    unique = set(travis['precinct'].unique())
    matches = unique & set(precincts['PCTKEY'].values)
    print(f"{year}: {len(travis)} rows, {len(unique)} unique precincts, {len(matches)} matching boundary")
    print(f"  Columns sample: {list(df.columns[:8])}")

2016: 0 rows, 0 unique precincts, 0 matching boundary
  Columns sample: ['year', 'stage', 'special', 'state', 'state_postal', 'state_fips', 'state_icpsr', 'county_name']
2018: 741 rows, 247 unique precincts, 0 matching boundary
  Columns sample: ['precinct', 'office', 'party_detailed', 'party_simplified', 'mode', 'votes', 'county_name', 'county_fips']


/Users/flyingcape/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3165: DtypeWarning: Columns (5,11,17) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


2022: 906 rows, 295 unique precincts, 0 matching boundary
  Columns sample: ['precinct', 'office', 'party_detailed', 'party_simplified', 'mode', 'votes', 'county_name', 'county_fips']


/Users/flyingcape/opt/anaconda3/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3165: DtypeWarning: Columns (5,11) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


2024: 823 rows, 300 unique precincts, 0 matching boundary
  Columns sample: ['precinct', 'office', 'party_detailed', 'party_simplified', 'mode', 'votes', 'county_name', 'county_fips']


In [18]:
# Check 2016 county names
df_2016 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2016.tab', 
                       sep=',', dtype={'precinct': str})
tx_2016 = df_2016[df_2016['state_postal'] == 'TX']
print("2016 TX county names sample:")
print(tx_2016['county_name'].unique()[:10])

# Check 2018 precinct format
df_2018 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2018.csv',
                       dtype={'precinct': str, 'county_fips': str})
travis_2018 = df_2018[df_2018['county_name'] == 'TRAVIS']
print("\n2018 Travis precinct sample:")
print(travis_2018['precinct'].head(10).tolist())

2016 TX county names sample:
['Anderson County' 'Andrews County' 'Angelina County' 'Aransas County'
 'Archer County' 'Armstrong County' 'Atascosa County' 'Austin County'
 'Bailey County' 'Bandera County']

2018 Travis precinct sample:
['4530103_8342', '4530103_8342', '4530103_8342', '4530104_8343', '4530104_8343', '4530104_8343', '4530105_8344', '4530105_8344', '4530105_8344', '4530106_8345']


In [19]:
files = {
    '2016': ('../data/raw/election_results/HOUSE_precinct_general_2016.tab', ',', 'Travis County', 'state_postal', None),
    '2018': ('../data/raw/election_results/HOUSE_precinct_general_2018.csv', ',', 'TRAVIS', 'state_po', '_'),
    '2022': ('../data/raw/election_results/HOUSE_precinct_general_2022.csv', ',', 'TRAVIS', 'state_po', '_'),
    '2024': ('../data/raw/election_results/HOUSE_precinct_general_2024.csv', ',', 'TRAVIS', 'state_po', '_'),
}

for year, (path, sep, county, state_col, split_char) in files.items():
    df = pd.read_csv(path, sep=sep, dtype={'precinct': str, 'county_fips': str}, low_memory=False)
    travis = df[df['county_name'] == county].copy()
    
    if split_char:
        travis['PCTKEY'] = travis['precinct'].str.split(split_char).str[0]
    else:
        travis['PCTKEY'] = travis['precinct']
    
    unique = set(travis['PCTKEY'].unique())
    matches = unique & set(precincts['PCTKEY'].values)
    print(f"{year}: {len(travis)} rows, {len(unique)} unique precincts, {len(matches)} matching boundary")

2016: 834 rows, 247 unique precincts, 247 matching boundary
2018: 741 rows, 247 unique precincts, 247 matching boundary
2022: 906 rows, 295 unique precincts, 193 matching boundary
2024: 823 rows, 300 unique precincts, 0 matching boundary


In [20]:
# Check 2024 precinct format
df_2024 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2024.csv',
                       dtype={'precinct': str, 'county_fips': str}, low_memory=False)
travis_2024 = df_2024[df_2024['county_name'] == 'TRAVIS'].copy()
travis_2024['PCTKEY'] = travis_2024['precinct'].str.split('_').str[0]
print("2024 PCTKEY sample:")
print(travis_2024['PCTKEY'].head(10).tolist())

# Check 2022 mismatches
df_2022 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2022.csv',
                       dtype={'precinct': str, 'county_fips': str}, low_memory=False)
travis_2022 = df_2022[df_2022['county_name'] == 'TRAVIS'].copy()
travis_2022['PCTKEY'] = travis_2022['precinct'].str.split('_').str[0]
only_2022 = set(travis_2022['PCTKEY'].unique()) - set(precincts['PCTKEY'].values)
print(f"\n2022 precincts not in boundary file (sample):")
print(sorted(only_2022)[:20])

2024 PCTKEY sample:
['2270100', '2270100', '2270101', '2270101', '2270102', '2270102', '2270103', '2270103', '2270104', '2270104']

2022 precincts not in boundary file (sample):
['4530100', '4530143', '4530144', '4530150A', '4530150B', '4530151A', '4530154A', '4530154B', '4530155', '4530162', '4530165', '4530170', '4530171', '4530172', '4530173', '4530174', '4530176', '4530177', '4530178', '4530180']


In [21]:
# Check what county names exist in 2024 for Travis
print("2024 county names containing 'travis' (case insensitive):")
print(df_2024[df_2024['county_name'].str.lower().str.contains('travis', na=False)]['county_name'].unique())

print("\n2024 PCTKEY starting with 453:")
travis_2024_453 = travis_2024[travis_2024['PCTKEY'].str.startswith('453')]
print(f"Rows: {len(travis_2024_453)}")
print(travis_2024_453['PCTKEY'].head(10).tolist())

2024 county names containing 'travis' (case insensitive):
['TRAVIS']

2024 PCTKEY starting with 453:
Rows: 0
[]


In [22]:
# Check 2024 Travis rows more carefully
df_2024 = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2024.csv',
                       dtype={'precinct': str, 'county_fips': str}, low_memory=False)
travis_2024 = df_2024[df_2024['county_name'] == 'TRAVIS'].copy()
travis_2024['PCTKEY'] = travis_2024['precinct'].str.split('_').str[0]

print("Unique PCTKEY prefixes:")
print(travis_2024['PCTKEY'].str[:3].value_counts())

print("\nSample rows:")
print(travis_2024[['precinct', 'county_fips', 'jurisdiction_name', 'jurisdiction_fips', 'candidate']].head(5).to_string())

Unique PCTKEY prefixes:
227    823
Name: PCTKEY, dtype: int64

Sample rows:
        precinct county_fips jurisdiction_name  jurisdiction_fips candidate
1111011  2270100       48453            TRAVIS            48453.0     CASAR
1111012  2270100       48453            TRAVIS            48453.0    WRIGHT
1111013  2270101       48453            TRAVIS            48453.0     CASAR
1111014  2270101       48453            TRAVIS            48453.0    WRIGHT
1111015  2270102       48453            TRAVIS            48453.0     CASAR


In [23]:
# Check if last 4 digits match
travis_2024['prec_suffix'] = travis_2024['PCTKEY'].str[-4:]
boundary_suffix = set(precincts['PCTKEY'].str[-4:].values)

matches_suffix = set(travis_2024['prec_suffix'].unique()) & boundary_suffix
print(f"Unique 2024 precincts: {travis_2024['PCTKEY'].nunique()}")
print(f"Matching on last 4 digits: {len(matches_suffix)}")
print(f"\nSample 2024 PKTKEYs: {sorted(travis_2024['PCTKEY'].unique())[:10]}")
print(f"Sample boundary PCTKEYs: {sorted(precincts['PCTKEY'].values)[:10]}")

Unique 2024 precincts: 300
Matching on last 4 digits: 193

Sample 2024 PKTKEYs: ['2270100', '2270101', '2270102', '2270103', '2270104', '2270105', '2270106', '2270107', '2270110', '2270111']
Sample boundary PCTKEYs: ['4530101', '4530102', '4530103', '4530104', '4530105', '4530106', '4530107', '4530108', '4530109', '4530110']


In [24]:
import zipfile
import geopandas as gpd

# Check 2022 boundary file
with zipfile.ZipFile('../data/raw/boundaries/precincts22g.zip', 'r') as z:
    print("2022 files:", z.namelist()[:5])

# Check 2024 boundary file  
with zipfile.ZipFile('../data/raw/boundaries/precincts24g.zip', 'r') as z:
    print("2024 files:", z.namelist()[:5])

2022 files: ['Precincts22G.cpg', 'Precincts22G.dbf', 'Precincts22G.prj', 'Precincts22G.sbn', 'Precincts22G.sbx']
2024 files: ['Precincts24G.cpg', 'Precincts24G.dbf', 'Precincts24G.prj', 'Precincts24G.sbn', 'Precincts24G.sbx']


In [25]:
# Load 2022 boundary
tx_2022_boundary = gpd.read_file('zip://../data/raw/boundaries/precincts22g.zip!Precincts22G.shp')
travis_2022_boundary = tx_2022_boundary[tx_2022_boundary['CNTY'] == 453].copy()
print(f"2022 Travis precincts: {len(travis_2022_boundary)}")
print(f"PCTKEY sample: {sorted(travis_2022_boundary['PCTKEY'].values)[:10]}")

# Load 2024 boundary
tx_2024_boundary = gpd.read_file('zip://../data/raw/boundaries/precincts24g.zip!Precincts24G.shp')
travis_2024_boundary = tx_2024_boundary[tx_2024_boundary['CNTY'] == 453].copy()
print(f"\n2024 Travis precincts: {len(travis_2024_boundary)}")
print(f"PCTKEY sample: {sorted(travis_2024_boundary['PCTKEY'].values)[:10]}")

2022 Travis precincts: 287
PCTKEY sample: ['4530100', '4530101', '4530102', '4530103', '4530104', '4530105', '4530106', '4530107', '4530110', '4530111']

2024 Travis precincts: 287
PCTKEY sample: ['4530100', '4530101', '4530102', '4530103', '4530104', '4530105', '4530106', '4530107', '4530110', '4530111']


In [26]:
# Check 2024 MEDSL against 2024 boundary
boundary_2024_unique = set(travis_2024_boundary['PCTKEY'].values)
medsl_2024_unique = set(travis_2024['PCTKEY'].unique())

print(f"2024 boundary precincts: {len(boundary_2024_unique)}")
print(f"2024 MEDSL precincts: {len(medsl_2024_unique)}")
print(f"Sample MEDSL 2024 PCTKEYs: {sorted(medsl_2024_unique)[:5]}")
print(f"Sample boundary 2024 PCTKEYs: {sorted(boundary_2024_unique)[:5]}")

# Try matching on last 4 digits
medsl_suffix = set(p[-4:] for p in medsl_2024_unique)
boundary_suffix = set(p[-4:] for p in boundary_2024_unique)
matches = medsl_suffix & boundary_suffix
print(f"\nMatching on last 4 digits: {len(matches)} of {len(boundary_2024_unique)}")

2024 boundary precincts: 287
2024 MEDSL precincts: 300
Sample MEDSL 2024 PCTKEYs: ['2270100', '2270101', '2270102', '2270103', '2270104']
Sample boundary 2024 PCTKEYs: ['4530100', '4530101', '4530102', '4530103', '4530104']

Matching on last 4 digits: 277 of 287


In [27]:
# Test the prefix replacement
travis_2024['PCTKEY_fixed'] = travis_2024['PCTKEY'].str.replace('^227', '453', regex=True)

matches_fixed = set(travis_2024['PCTKEY_fixed'].unique()) & boundary_2024_unique
only_medsl = set(travis_2024['PCTKEY_fixed'].unique()) - boundary_2024_unique
only_boundary = boundary_2024_unique - set(travis_2024['PCTKEY_fixed'].unique())

print(f"After prefix fix:")
print(f"Matching: {len(matches_fixed)}")
print(f"Only in MEDSL: {len(only_medsl)}")
print(f"Only in boundary: {len(only_boundary)}")

After prefix fix:
Matching: 277
Only in MEDSL: 23
Only in boundary: 10


In [28]:
# Check 2022 MEDSL against 2022 boundary
df_2022_medsl = pd.read_csv('../data/raw/election_results/HOUSE_precinct_general_2022.csv',
                       dtype={'precinct': str, 'county_fips': str}, low_memory=False)
travis_2022_medsl = df_2022_medsl[df_2022_medsl['county_name'] == 'TRAVIS'].copy()
travis_2022_medsl['PCTKEY'] = travis_2022_medsl['precinct'].str.split('_').str[0]

# Remove letter suffixes
travis_2022_medsl['PCTKEY_clean'] = travis_2022_medsl['PCTKEY'].str.replace('[A-Z]$', '', regex=True)

boundary_2022_unique = set(travis_2022_boundary['PCTKEY'].values)
medsl_2022_unique = set(travis_2022_medsl['PCTKEY_clean'].unique())

matches_2022 = medsl_2022_unique & boundary_2022_unique
only_medsl_2022 = medsl_2022_unique - boundary_2022_unique
only_boundary_2022 = boundary_2022_unique - medsl_2022_unique

print(f"2022 boundary precincts: {len(boundary_2022_unique)}")
print(f"2022 MEDSL precincts: {len(medsl_2022_unique)}")
print(f"Matching: {len(matches_2022)}")
print(f"Only in MEDSL: {len(only_medsl_2022)}")
print(f"Only in boundary: {len(only_boundary_2022)}")

2022 boundary precincts: 287
2022 MEDSL precincts: 286
Matching: 286
Only in MEDSL: 0
Only in boundary: 1
